In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load dataset
df = pd.read_csv("California_Houses.csv")
df = df.fillna(df.median())  # fill missing values

# we remove this column from the dataset because it is the target we want to predict.
# .values → converts the DataFrame into a NumPy array for faster and easier calculations.
X = df.drop("Median_House_Value", axis=1).values

# Target variable -> the values the model is supposed to learn and predict.
#.values.reshape(-1,1) → reshapes it from (20640,) to (20640, 1) so it is compatible with matrix operations.
y = df["Median_House_Value"].values.reshape(-1, 1)

# Add bias column for manual regression
X = np.hstack((np.ones((X.shape[0], 1)), X))

# Split dataset: 70% train, 15% val, 15% test
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42)

# Normalize features (except bias column)
scaler = StandardScaler()
X_train[:,1:] = scaler.fit_transform(X_train[:,1:])
X_val[:,1:] = scaler.transform(X_val[:,1:])
X_test[:,1:] = scaler.transform(X_test[:,1:])

In [ ]:
# Normal Equation function
def normal_eq(X, y):
    # Normal Equation: w = (X^T X)^(-1) X^T 
    return np.linalg.inv(X.T @ X) @ X.T @ y

# Calculate weights
w_linear = normal_eq(X_train, y_train)

# Predictions
y_pred_manual = X_test @ w_linear

In [3]:
def ridge_normal_eq(X, y, lam):
    n, f = X.shape
    I = np.identity(f)
    return np.linalg.inv(X.T @ X + lam * n * I) @ X.T @ y

w_ridge_manual = ridge_normal_eq(X_train, y_train, lam=0.001)
y_pred_ridge_manual = X_test @ w_ridge_manual

In [ ]:
# Lasso using gradient descent
def lasso_gd(X, y, lam=0.001, alpha=0.01, epochs=1000):
    n, f = X.shape
    w = np.zeros((f,1))  # initialize weights
    
    for i in range(epochs):
        grad = X.T @ (X @ w - y)  # gradient of MSE
        # update with L1 regularization
        w = w - (alpha / n) * (grad + lam * np.sign(w)) 
    return w

# Calculate Lasso weights
w_lasso_manual = lasso_gd(X_train, y_train, lam=0.001, alpha=0.01, epochs=5000)

# Predictions
y_pred_lasso_manual = X_test @ w_lasso_manual

In [4]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Linear Regression
linear_model = LinearRegression()
linear_model.fit(X_train[:,1:], y_train)  # skip bias column for sklearn
y_pred_sklearn_linear = linear_model.predict(X_test[:,1:])

# Ridge Regression
ridge_model = Ridge(alpha=0.001)
ridge_model.fit(X_train[:,1:], y_train)
y_pred_sklearn_ridge = ridge_model.predict(X_test[:,1:])

# Lasso Regression
lasso_model = Lasso(alpha=0.001, max_iter=10000)
lasso_model.fit(X_train[:,1:], y_train)
y_pred_sklearn_lasso = lasso_model.predict(X_test[:,1:])

In [ ]:
# Compare all models
results = pd.DataFrame({
    "Model": ["Manual Linear", "Sklearn Linear", "Manual Ridge", "Sklearn Ridge", "Manual Lasso", "Sklearn Lasso"],
    "MSE": [
        mean_squared_error(y_test, y_pred_manual),
        mean_squared_error(y_test, y_pred_sklearn_linear),
        mean_squared_error(y_test, y_pred_ridge_manual),
        mean_squared_error(y_test, y_pred_sklearn_ridge),
        mean_squared_error(y_test, y_pred_lasso_manual),
        mean_squared_error(y_test, y_pred_sklearn_lasso)
    ],
    "MAE": [
        mean_absolute_error(y_test, y_pred_manual),
        mean_absolute_error(y_test, y_pred_sklearn_linear),
        mean_absolute_error(y_test, y_pred_ridge_manual),
        mean_absolute_error(y_test, y_pred_sklearn_ridge),
        mean_absolute_error(y_test, y_pred_lasso_manual),
        mean_absolute_error(y_test, y_pred_sklearn_lasso)
    ]
})

results

,Model,MSE,MAE
0,Manual Linear,4.400953e+09,48782.031081
1,Sklearn Linear,4.400953e+09,48782.031081
2,Manual Ridge,4.399464e+09,48805.389393
3,Sklearn Ridge,4.400953e+09,48782.033312
4,Manual Lasso,4.427993e+09,49173.967087
5,Sklearn Lasso,4.400953e+09,48782.031545
